# AI Desktop Pet Client - Demo Notebook

This notebook demonstrates the core functionality of the AI Desktop Pet client application.

## Table of Contents
1. [Overview](#Overview)
2. [Client-Server Communication](#Client-Server-Communication)
3. [Live2D Model Integration](#Live2D-Model-Integration)
4. [WSL2 Command Execution](#WSL2-Command-Execution)
5. [Mode Switching](#Mode-Switching)

## Overview

The AI Desktop Pet client is built with:
- **Electron**: Desktop application framework
- **Vue.js**: Frontend framework
- **Live2D**: 2D character animation
- **Express**: Local server for WSL2 commands

In [ ]:
import requests
import json

# Configuration
SERVER_URL = 'http://localhost:8000'
LOCAL_SERVER_URL = 'http://localhost:3000'

print(f"Server URL: {SERVER_URL}")
print(f"Local Server URL: {LOCAL_SERVER_URL}")

## Client-Server Communication

### Creating a Session

First, we need to create a session to maintain conversation context.

In [ ]:
# Create a new session
response = requests.post(f"{SERVER_URL}/api/v1/session")
session_id = response.json()['session_id']

print(f"Session ID: {session_id}")

### Sending a Chat Message

Now let's send a message to the AI and get a response.

In [ ]:
# Send a chat message
message = "Hello, introduce yourself!"

response = requests.post(
    f"{SERVER_URL}/api/v1/chat",
    json={
        "session_id": session_id,
        "message": message,
        "stream": False
    }
)

data = response.json()
print(f"AI Response: {data.get('text', 'No response')}")
print(f"Emotion: {data.get('emotion', 'unknown')}")
print(f"Action: {data.get('action', 'idle')}")

## Live2D Model Integration

The client uses Live2D models to display animated characters. Here's how the emotion mapping works:

In [ ]:
# Emotion to Live2D mapping
emotion_map = {
    'happy': {'expression': 'exp_01', 'motion': 'special_01'},
    'sad': {'expression': 'exp_02', 'motion': 'mtn_02'},
    'angry': {'expression': 'exp_03', 'motion': 'mtn_03'},
    'playful': {'expression': 'exp_04', 'motion': 'special_02'},
    'shy': {'expression': 'exp_06', 'motion': 'mtn_01'},
    'neutral': {'expression': 'exp_01', 'motion': 'mtn_01'}
}

print("Emotion Mapping:")
for emotion, config in emotion_map.items():
    print(f"  {emotion}: expression={config['expression']}, motion={config['motion']}")

## WSL2 Command Execution

The client uses a local Express server to execute WSL2 commands safely.

In [ ]:
# Execute a WSL2 command via local server
command = "ls -la"
timeout = 30

try:
    response = requests.post(
        f"{LOCAL_SERVER_URL}/execute",
        json={
            "command": command,
            "timeout": timeout
        }
    )
    
    result = response.json()
    print(f"Command: {result['command']}")
    print(f"Exit Code: {result['returncode']}")
    print(f"Output:\n{result['stdout']}")
    if result['stderr']:
        print(f"Error:\n{result['stderr']}")
        
except requests.exceptions.ConnectionError:
    print("Local server not running. Please start it first.")

## Mode Switching

The client supports two main modes:
1. **Chat Mode**: Normal conversation with AI
2. **WSL2 Mode**: Execute WSL2 commands via AI

In [ ]:
# Example: Chat Mode
print("=== Chat Mode ===")
response = requests.post(
    f"{SERVER_URL}/api/v1/chat",
    json={
        "session_id": session_id,
        "message": "What's the weather like today?",
        "mode": "chat",
        "stream": False
    }
)
data = response.json()
print(f"Response: {data.get('text', 'No response')}")

# Example: WSL2 Mode
print("\n=== WSL2 Mode ===")
response = requests.post(
    f"{SERVER_URL}/api/v1/chat",
    json={
        "session_id": session_id,
        "message": "List files in current directory",
        "mode": "wsl2",
        "stream": False
    }
)
data = response.json()
print(f"WSL2 Command: {data.get('wsl2_command', 'No command')}")
print(f"Response: {data.get('text', 'No response')}")

## Summary

This notebook demonstrated:
- Creating sessions and sending chat messages
- Understanding emotion-to-Live2D mapping
- Executing WSL2 commands via local server
- Switching between chat and WSL2 modes

For more information, see the project documentation.